 # Network Analysis



 This notebook analyses the structure of the inferred gene regulatory networks

 (GRNs) stored in `adata.varp['W_<cluster>']`.



 Topics covered:

 1. Network correlation dendrograms

 2. Symmetricity of W matrices

 3. Network centrality (degree, betweenness, eigenvector)

 4. Eigenanalysis of interaction matrices

 5. GRN visualisation

 ## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as scp
from scipy.spatial.distance import squareform 
import scHopfield as sch

DATA_PATH    = '/path/to/data/'
DATASET_FILE = 'hematopoiesis.h5ad'
MODEL_FILE   = 'model.h5sch'

CLUSTER_KEY = 'cell_type'
SPLICED_KEY  = 'M_t'

CELL_TYPE_ORDER = ['Meg', 'Ery', 'MEP-like', 'HSC', 'GMP-like', 'Mon', 'Bas', 'Neu']

adata = sc.read_h5ad(DATA_PATH + DATASET_FILE)
sch.tl.load_model(adata, MODEL_FILE)
print(adata)

# Cell-type colour palette
colors = {ct: f'C{i}' for i, ct in enumerate(CELL_TYPE_ORDER)}


 ## 3.1 Network Correlation Dendrograms



 Compute pairwise similarity between cell-type networks using the RV coefficient

 and Pearson / Hamming distances.

In [ ]:
# Cell-type similarity based on expression covariance (RV score)
sch.tl.celltype_correlation(
    adata,
    spliced_key=SPLICED_KEY,
    cluster_key=CLUSTER_KEY
)

cells_correlation = adata.uns['scHopfield']['celltype_correlation']

fig, ax = plt.subplots(figsize=(10, 4), tight_layout=True)
Z = scp.cluster.hierarchy.linkage(squareform(1 - cells_correlation), 'complete')
scp.cluster.hierarchy.dendrogram(Z, labels=cells_correlation.index, ax=ax)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_title('Cell-type RV score dendrogram')
plt.show()


In [ ]:
# Network-structure similarity (W matrix correlations)
sch.tl.network_correlations(adata, cluster_key=CLUSTER_KEY)

pearson    = adata.uns['scHopfield']['network_correlations']['pearson']
hamming    = adata.uns['scHopfield']['network_correlations']['hamming']
pearson_bin = adata.uns['scHopfield']['network_correlations']['pearson_bin']

fig, ax = plt.subplots(figsize=(9, 3), tight_layout=True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.get_yaxis().set_visible(False)

Z = scp.cluster.hierarchy.linkage(squareform(1 - pearson), 'complete')
scp.cluster.hierarchy.dendrogram(Z, labels=pearson.index, ax=ax)
ax.set_title('Network Pearson similarity dendrogram')
plt.show()


 ## 3.2 Symmetricity Analysis



 A symmetric W ≈ equilibrium system; a skew-symmetric W ≈ oscillatory system.

 The symmetricity score ranges from −1 (fully skew-symmetric) to +1 (fully

 symmetric).

In [ ]:
def symmetricity(A, norm=2):
    """Ratio of symmetric to total Frobenius norm."""
    S  = np.linalg.norm((A + A.T) / 2, ord=norm)
    As = np.linalg.norm((A - A.T) / 2, ord=norm)
    return (S - As) / (S + As)

genes_used = adata.var['use_for_dynamics'].values
W = {
    cluster: adata.varp[f'W_{cluster}'][genes_used][:, genes_used]
    for cluster in CELL_TYPE_ORDER
}

syms = np.array([symmetricity(W[k]) for k in CELL_TYPE_ORDER])

fig, ax = plt.subplots(figsize=(5, 4), tight_layout=True)
ax.scatter(
    range(len(CELL_TYPE_ORDER)), syms, s=200, marker='*',
    c=[colors[k] for k in CELL_TYPE_ORDER]
)
ax.set_xticks(range(len(CELL_TYPE_ORDER)))
ax.set_xticklabels(CELL_TYPE_ORDER, rotation=-30)
ax.set_ylabel('Symmetricity')
ax.set_title('Symmetricity of W matrices across cell types')
plt.show()


 ## 3.3 Network Centrality



 Three complementary centrality measures:

 - **degree_centrality_out**: fraction of outgoing edges

 - **betweenness_centrality**: fraction of shortest paths through node

 - **eigenvector_centrality**: influence weighted by neighbour importance

In [ ]:
sch.tl.compute_network_centrality(
    adata,
    cluster_key=CLUSTER_KEY,
    threshold_number=40000   # keep top-N edges per cluster
)


In [ ]:
# Ranked gene plots for each metric
fig, axes = plt.subplots(1, 3, figsize=(15, 10))

metrics = ['degree_centrality_out', 'betweenness_centrality', 'eigenvector_centrality']
for ax, metric in zip(axes, metrics):
    sch.pl.plot_network_centrality_rank(
        adata,
        metric=metric,
        clusters=CELL_TYPE_ORDER,
        cluster_key=CLUSTER_KEY,
        n_genes=20,
        colors=colors,
        ax=ax
    )
plt.tight_layout()
plt.show()


In [ ]:
# Top-gene tables
df_betweenness = sch.tl.get_top_genes_table(
    adata,
    metric='betweenness_centrality',
    cluster_key=CLUSTER_KEY,
    n_genes=20,
    order=CELL_TYPE_ORDER
)
print("Betweenness centrality top genes:")
print(df_betweenness)

df_degree = sch.tl.get_top_genes_table(
    adata,
    metric='degree_centrality_out',
    cluster_key=CLUSTER_KEY,
    n_genes=20,
    order=CELL_TYPE_ORDER
)
print("\nDegree centrality top genes:")
print(df_degree)


In [ ]:
# Betweenness vs degree scatter (high betweenness, low degree = bottleneck genes)
fig = sch.pl.plot_centrality_scatter(
    adata,
    x_metric='betweenness_centrality',
    y_metric='degree_centrality_all',
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=colors,
    n_top_genes=3,
    filter_threshold=('degree_centrality_all', '<', 0.5),
    figsize=(20, 10)
)
plt.show()


 ## 3.4 Eigenanalysis



 Eigendecomposition of each cluster's W matrix reveals:

 - Dominant regulatory directions (top eigenvector genes)

 - Stability indicators (sign of leading eigenvalue)

In [ ]:
sch.tl.compute_eigenanalysis(adata, cluster_key=CLUSTER_KEY)


In [ ]:
# Comprehensive grid: spectrum + max/min eigenvector gene loadings
fig = sch.pl.plot_eigenanalysis_grid(
    adata,
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=colors,
    n_genes=10,
    figsize=(16, 4 * len(CELL_TYPE_ORDER))
)
plt.show()


In [ ]:
# Table of top genes from extreme eigenvectors
df_eigenanalysis = sch.tl.get_eigenanalysis_table(
    adata,
    cluster_key=CLUSTER_KEY,
    n_genes=20,
    order=CELL_TYPE_ORDER
)
print(df_eigenanalysis.head(10))


In [ ]:
# Eigenvalue spectra for all clusters overlaid
fig, ax = plt.subplots(figsize=(10, 10))
sch.pl.plot_eigenvalue_spectrum(
    adata,
    clusters=CELL_TYPE_ORDER,
    cluster_key=CLUSTER_KEY,
    colors=colors,
    highlight_extremes=True,
    ax=ax
)
plt.show()


In [ ]:
# Leading eigenvector per cluster — extreme eigenvalues summary
print("Extreme eigenvalues per cluster:")
for cluster in CELL_TYPE_ORDER:
    evals = adata.uns['scHopfield']['eigenanalysis'][f'eigenvalues_{cluster}']
    max_ev = evals[np.argmax(evals.real)]
    min_ev = evals[np.argmin(evals.real)]
    print(f"  {cluster:15s} | Max Re(λ): {max_ev.real:8.3f} | Min Re(λ): {min_ev.real:8.3f}")


 ## 3.5 GRN Visualisation

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

colors_graph = ['blue', 'lightgray', 'red']
custom_cmap  = LinearSegmentedColormap.from_list('custom_cmap', list(zip([0, 0.5, 1], colors_graph)))

score = 'degree_centrality_out'
topn  = 50

fig, axs = plt.subplots(4, 2, figsize=(20, 40), tight_layout=True)

for cluster, ax in zip(CELL_TYPE_ORDER, axs.flat):
    ax.axis('off')
    ax.set_title(f'{score.replace("_", " ").capitalize()} — {cluster}')
    sch.pl.plot_grn_network(
        adata,
        cluster=cluster,
        topn=topn,
        score_size=score,
        ax=ax,
        cmap=custom_cmap
    )

plt.show()
